<a href="https://colab.research.google.com/github/sorotdaniel/portfolio-soro/blob/main/regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Données d'une centrale thermique\
AT = Température ambiante

V = Vitesse d’échappement

AP = Pression atmosphérique

RH = Humidité relative

PE = Puissance électrique produite

# 1- Regression linéaire
$$PE=β_0​+β_1​AT+β_2​V+β_3​AP+β_4​RH+ε$$

	​


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

In [ ]:
df = pd.read_csv("/content/Energie.csv", sep=';', decimal=',')
df.head()


,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [ ]:
df.describe()

,AT,V,AP,RH,PE
count,9568.000000,9568.000000,9568.000000,9568.000000,9568.000000
mean,19.651231,54.305804,1013.259078,73.308978,454.365009
std,7.452473,12.707893,5.938784,14.600269,17.066995
min,1.810000,25.360000,992.890000,25.560000,420.260000
25%,13.510000,41.740000,1009.100000,63.327500,439.750000
50%,20.345000,52.080000,1012.940000,74.975000,451.550000
75%,25.720000,66.540000,1017.260000,84.830000,468.430000
max,37.110000,81.560000,1033.300000,100.160000,495.760000


In [ ]:
corr= df.corr()
corr

,AT,V,AP,RH,PE
AT,1.000000,0.844107,-0.507549,-0.542535,-0.948128
V,0.844107,1.000000,-0.413502,-0.312187,-0.869780
AP,-0.507549,-0.413502,1.000000,0.099574,0.518429
RH,-0.542535,-0.312187,0.099574,1.000000,0.389794
PE,-0.948128,-0.869780,0.518429,0.389794,1.000000


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. CHARGER ET PREPARER
X = df.drop('PE', axis=1)
y = df['PE']

# 2. SPLIT TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42)

# 3. CREER ET ENTRAINER
model = LinearRegression()
model.fit(X_train, y_train)

# 4. PREDIRE
y_pred = model.predict(X_test)

# 5. EVALUER
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f} MW " )
print(f"R2: {r2:.3f}")

RMSE: 4.43 MW 
R2: 0.931


## Récupérer les coéfficients

In [ ]:
#Recuperer intercept et coefficients
intercept = model.intercept_
coefficients = model.coef_

# Creer DataFrame pour visualisation
coef_df = pd.DataFrame({
  'Variable': X.columns,
  'Coefficient': coefficients,
  'Impact_Absolu': np.abs(coefficients)
}).sort_values('Impact_Absolu', ascending=False)

print(f"Intercept: {intercept:.2f} MW\n")
print(coef_df)

Intercept: 458.60 MW

  Variable  Coefficient  Impact_Absolu
0       AT    -1.976966       1.976966
1        V    -0.234769       0.234769
3       RH    -0.158146       0.158146
2       AP     0.058253       0.058253


# Ridge et Lasso

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# RIDGE
ridge = Ridge(alpha=0.05)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

# LASSO
lasso = Lasso(alpha=0.05)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

# ELASTICNET
elastic = ElasticNet(alpha=0.05, l1_ratio=0.05)
elastic.fit(X_train, y_train)
y_pred_elastic = elastic.predict(X_test)

# Comparer coefficients
print("Coefficients Ridge:", ridge.coef_)
print("Coefficients Lasso:", lasso.coef_)
print("Coefficients elastic:", elastic.coef_)
# Lasso peut avoir des coefficients = 0

# Evaluer
print(f"RMSE Ridge: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.2f}")
print(f"RMSE Lasso: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.2f}")

Coefficients Ridge: [-1.97696463 -0.23476932  0.05825332 -0.15814552]
Coefficients Lasso: [-1.97481723 -0.23546283  0.05741354 -0.1574658 ]
Coefficients elastic: [-1.96762607 -0.23831902  0.06051651 -0.15655829]
RMSE Ridge: 4.43
RMSE Lasso: 4.43


## Polynomiale

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Créer des caractéristiques polynomiale
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X)

model = LinearRegression()
model.fit(X_poly, y)

print(f"Intercept: {(model.intercept_):.2f}")
print(f"Coefficients:  {model.coef_}")

Intercept: -7503.73
Coefficients:  [ 0.00000000e+00 -5.79689341e+00 -3.04429835e+00  1.56283883e+01
  3.91936109e+00  1.63263830e-02  1.19701357e-02  3.11491053e-03
 -6.08210972e-03 -1.25988776e-03  2.57372866e-03  5.05456778e-04
 -7.61653213e-03 -3.61850092e-03 -1.92565185e-03]


In [ ]:
# 2. SPLIT TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(
X_poly, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# RIDGE
ridge = Ridge(alpha=1, max_iter=10000)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

# LASSO
lasso = Lasso(alpha=1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

# ELASTICNET
elastic = ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=10000)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_test_scaled)

# Comparer coefficients
print("Coefficients Ridge:", ridge.coef_)
print("Coefficients Lasso:", lasso.coef_)
print("Coefficients elastic:", elastic.coef_)
# Lasso peut avoir des coefficients = 0

# Evaluer
print(f"RMSE Ridge: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.2f}")
print(f"RMSE Lasso: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.2f}")

Coefficients Ridge: [-14.76026533  -2.99109247   0.34632267  -2.2999704 ]
Coefficients Lasso: [-1.20521369e+01 -3.60363366e+00  2.34571141e-01 -6.02873057e-03]
Coefficients elastic: [-8.39404156e+00 -5.48016846e+00  1.64746484e+00  1.86940059e-03]
RMSE Ridge: 4.43
RMSE Lasso: 4.94
